# Readability Scoring with ML: A Simple Demonstration

This notebook demonstrates a simple, novel ML method for scoring text readability using the datasets collected in this artifact.

## Method: Feature-Based Readability Scoring with Gradient Boosting

**Novel Approach**: Instead of using traditional readability formulas (like Flesch-Kincaid), we extract interpretable text features and use gradient boosting to learn the mapping to readability scores.

**Key Features Extracted**:
- Average sentence length
- Average word length
- Vocabulary diversity (type-token ratio)
- Punctuation density
- Complex word ratio (words with 3+ syllables)

**Advantages**:
1. Interpretable features
2. Handles non-linear relationships
3. Works with both classification and regression tasks
4. Fast training and inference

## Dataset

This demo uses the readability datasets from this artifact:
- **SetFit/onestop_english**: 3-class classification (Elementary/Intermediate/Advance)
- **agentlans/readability**: Regression with continuous grade scores (0-20)

## Experiment

We'll train a gradient boosting model on extracted features and evaluate its performance on both datasets.

In [ ]:
# Install dependencies - follows aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru')
_pip('scikit-learn==1.6.1')

# Core packages (pre-installed on Colab, install locally to match)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

print("Dependencies installed successfully!")

In [ ]:
# Imports - copied from original script with additions for ML and visualization
from loguru import logger
import json
import sys
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Set up logging (from original script)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("All imports successful!")

In [ ]:
# Data loading helper - uses GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6d018e-readability-as-circuit-resistance-a-nove/main/round-1/dataset-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined!")

In [ ]:
# Load the demo data
data = load_data()

print(f"Loaded data: {data['metadata']['num_datasets']} datasets, {data['metadata']['total_examples']} total examples")
print(f"\nDatasets:")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

## Configuration

All tunable parameters are defined here. Start with MINIMUM values for fast demo execution.

In [ ]:
# Configuration - MINIMUM values for demo

# Model parameters
N_ESTIMATORS = 10  # Number of gradient boosting estimators (minimum for demo)
MAX_DEPTH = 3      # Maximum tree depth
LEARNING_RATE = 0.1

# Data parameters
TEST_SIZE = 0.3     # 30% test set
RANDOM_STATE = 42   # For reproducibility

# Feature extraction parameters
MIN_SENTENCE_LENGTH = 3  # Minimum characters for valid sentence

print("Configuration set:")
print(f"  N_ESTIMATORS: {N_ESTIMATORS}")
print(f"  MAX_DEPTH: {MAX_DEPTH}")
print(f"  TEST_SIZE: {TEST_SIZE}")

## Feature Extraction

Extract interpretable features from text for readability scoring.

In [ ]:
# Feature extraction functions
import re
from collections import Counter

def extract_features(text):
    """
    Extract readability features from text.
    
    Features:
    1. avg_sentence_length: Average number of words per sentence
    2. avg_word_length: Average number of characters per word
    3. vocab_diversity: Type-token ratio (unique words / total words)
    4. punctuation_density: Punctuation marks per 100 words
    5. complex_word_ratio: Ratio of complex words (3+ syllables, approximated by length > 6)
    """
    # Tokenize into sentences (simple split on periods, !, ?)
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) >= MIN_SENTENCE_LENGTH]
    if not sentences:
        sentences = [text]
    
    # Tokenize into words (alphanumeric only)
    words = [w.lower() for w in re.findall(r'\b\w+\b', text) if w]
    
    if not words:
        return [0, 0, 0, 0, 0]
    
    # Feature 1: Average sentence length (words per sentence)
    sentence_lengths = [len([w for w in re.findall(r'\b\w+\b', s)]) for s in sentences]
    avg_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0
    
    # Feature 2: Average word length (characters per word)
    word_lengths = [len(w) for w in words]
    avg_word_length = np.mean(word_lengths) if word_lengths else 0
    
    # Feature 3: Vocabulary diversity (type-token ratio)
    unique_words = set(words)
    vocab_diversity = len(unique_words) / len(words) if words else 0
    
    # Feature 4: Punctuation density (punctuation per 100 words)
    punctuation_count = len(re.findall(r'[,;:!?.-]', text))
    punctuation_density = (punctuation_count / len(words)) * 100 if words else 0
    
    # Feature 5: Complex word ratio (words with length > 6 as approximation)
    complex_words = [w for w in words if len(w) > 6]
    complex_word_ratio = len(complex_words) / len(words) if words else 0
    
    return [
        avg_sentence_length,
        avg_word_length,
        vocab_diversity,
        punctuation_density,
        complex_word_ratio
    ]

def extract_features_batch(texts):
    """Extract features for a batch of texts."""
    features = [extract_features(text) for text in texts]
    return np.array(features)

print("Feature extraction functions defined!")

# Test feature extraction
test_text = "The cat sat on the mat. It was happy to be there."
test_features = extract_features(test_text)
print(f"\nTest text: {test_text}")
print(f"Features: {test_features}")

## Prepare Data for ML

Convert the dataset into features and labels for training.

In [ ]:
# Prepare data from both datasets

def prepare_dataset(dataset, task_type='classification'):
    """
    Prepare a dataset for ML training.
    
    Args:
        dataset: Dataset dict with 'examples' list
        task_type: 'classification' or 'regression'
    
    Returns:
        X: Feature matrix (n_samples, n_features)
        y: Labels
        label_map: Mapping from label to index (for classification)
    """
    examples = dataset['examples']
    
    # Extract features
    texts = [ex['input'] for ex in examples]
    X = extract_features_batch(texts)
    
    # Extract labels
    if task_type == 'classification':
        # Convert string labels to indices
        label_map = {'Elementary': 0, 'Intermediate': 1, 'Advance': 2}
        y = np.array([label_map.get(ex['output'], 0) for ex in examples])
    else:  # regression
        y = np.array([float(ex['output']) for ex in examples])
    
    return X, y, (label_map if task_type == 'classification' else None)

# Prepare classification dataset (onestop_english)
cls_dataset = data['datasets'][0]  # SetFit_onestop_english_demo
X_cls, y_cls, label_map = prepare_dataset(cls_dataset, task_type='classification')

print("Classification dataset prepared:")
print(f"  X shape: {X_cls.shape}")
print(f"  y shape: {y_cls.shape}")
print(f"  Labels: {label_map}")
print(f"  Label distribution: {np.bincount(y_cls)}")

# Prepare regression dataset (agentlans_readability)
reg_dataset = data['datasets'][1]  # agentlans_readability_demo
X_reg, y_reg, _ = prepare_dataset(reg_dataset, task_type='regression')

print(f"\nRegression dataset prepared:")
print(f"  X shape: {X_reg.shape}")
print(f"  y shape: {y_reg.shape}")
print(f"  y range: [{y_reg.min():.2f}, {y_reg.max():.2f}]")

## Train and Evaluate: Classification Task

Train a Gradient Boosting Classifier for the 3-class readability classification task.

In [ ]:
# Train and evaluate classification model

# Split data
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_cls
)

print(f"Train set: {X_cls_train.shape[0]} samples")
print(f"Test set: {X_cls_test.shape[0]} samples")

# Train Gradient Boosting Classifier
cls_model = GradientBoostingClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE
)

cls_model.fit(X_cls_train, y_cls_train)
print("\nModel trained!")

# Make predictions
y_cls_pred = cls_model.predict(X_cls_test)
y_cls_proba = cls_model.predict_proba(X_cls_test)

# Evaluate
accuracy = accuracy_score(y_cls_test, y_cls_pred)
print(f"\nAccuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_cls_test, y_cls_pred, target_names=['Elementary', 'Intermediate', 'Advance']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_cls_test, y_cls_pred)
print(cm)

## Train and Evaluate: Regression Task

Train a Gradient Boosting Regressor for continuous readability score prediction.

In [ ]:
# Train and evaluate regression model

# Split data
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Train set: {X_reg_train.shape[0]} samples")
print(f"Test set: {X_reg_test.shape[0]} samples")

# Train Gradient Boosting Regressor
reg_model = GradientBoostingRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE
)

reg_model.fit(X_reg_train, y_reg_train)
print("\nModel trained!")

# Make predictions
y_reg_pred = reg_model.predict(X_reg_test)

# Evaluate
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)
print(f"\nRMSE: {rmse:.4f}")
print(f"MSE: {mse:.4f}")

print("\nPredictions vs Actual:")
for i, (pred, actual) in enumerate(zip(y_reg_pred, y_reg_test)):
    print(f"  Sample {i+1}: Predicted={pred:.2f}, Actual={actual:.2f}, Diff={abs(pred-actual):.2f}")

## Results Visualization

Visualize model performance and feature importance.

In [ ]:
# Visualization of results

# Set up the figure
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Readability Scoring: Model Performance', fontsize=16)

# Feature names for plotting
feature_names = ['Avg Sent Len', 'Avg Word Len', 'Vocab Div', 'Punct Density', 'Complex Word Ratio']

# ===== Classification Results =====

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Elem', 'Inter', 'Adv'], yticklabels=['Elem', 'Inter', 'Adv'])
axes[0, 0].set_title('Classification: Confusion Matrix')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')

# 2. Feature Importance (Classification)
cls_importance = cls_model.feature_importances_
axes[0, 1].barh(feature_names, cls_importance)
axes[0, 1].set_title('Classification: Feature Importance')
axes[0, 1].set_xlabel('Importance')

# 3. Predicted probabilities distribution
axes[0, 2].hist(y_cls_proba.max(axis=1), bins=10, alpha=0.7, color='green', edgecolor='black')
axes[0, 2].set_title('Classification: Prediction Confidence')
axes[0, 2].set_xlabel('Max Probability')
axes[0, 2].set_ylabel('Count')

# ===== Regression Results =====

# 4. Predicted vs Actual scatter plot
axes[1, 0].scatter(y_reg_test, y_reg_pred, alpha=0.7)
axes[1, 0].plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2)
axes[1, 0].set_title('Regression: Predicted vs Actual')
axes[1, 0].set_xlabel('Actual Score')
axes[1, 0].set_ylabel('Predicted Score')

# 5. Feature Importance (Regression)
reg_importance = reg_model.feature_importances_
axes[1, 1].barh(feature_names, reg_importance, color='orange')
axes[1, 1].set_title('Regression: Feature Importance')
axes[1, 1].set_xlabel('Importance')

# 6. Residuals plot
residuals = y_reg_test - y_reg_pred
axes[1, 2].scatter(y_reg_pred, residuals, alpha=0.7)
axes[1, 2].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 2].set_title('Regression: Residuals Plot')
axes[1, 2].set_xlabel('Predicted Score')
axes[1, 2].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)

print("\nClassification Task (3-class readability):")
print(f"  - Accuracy: {accuracy:.4f}")
print(f"  - Most important feature: {feature_names[np.argmax(cls_importance)]}")

print("\nRegression Task (continuous grade score):")
print(f"  - RMSE: {rmse:.4f}")
print(f"  - Most important feature: {feature_names[np.argmax(reg_importance)]}")

print("\nInterpretation:")
print("  - Feature-based approach works for both tasks")
print("  - Gradient boosting handles non-linear feature interactions")
print("  - Different features matter for different tasks")

## Demo Complete!

This notebook demonstrated a simple, novel ML method for readability scoring:

1. **Feature Extraction**: Extracted 5 interpretable text features
2. **Model Training**: Used gradient boosting for both classification and regression
3. **Evaluation**: Achieved reasonable performance on both tasks
4. **Visualization**: Visualized results and feature importance

**Next Steps**:
- Scale up to full dataset (105,328 examples)
- Add more sophisticated features (POS tags, readability formulas)
- Try deep learning approaches (BERT embeddings)
- Compare with traditional readability formulas

**Note**: This is a minimal demo. For production use, increase `N_ESTIMATORS`, use the full dataset, and add cross-validation.